In [ ]:
# Find this checkout from a notebook or a kernel launched at the workspace root.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents, _here / 'cic17-cic18-transfer-study']
if os.environ.get('NIDS_STUDY_ROOT'):
    _candidates.insert(0, Path(os.environ['NIDS_STUDY_ROOT']).expanduser().resolve())
REPO_ROOT = next((p for p in _candidates if (p / 'src/nids_transfer').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this checkout or set NIDS_STUDY_ROOT.')
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
from nids_transfer.paths import study_paths
DATA_DIR, ARTIFACT_DIR = study_paths(REPO_ROOT)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_DIR)


# Dataset comparison — feature preparation

Scan the full compatible CIC2017 and CIC2018 datasets with DuckDB. Only small summaries are loaded into pandas; dataset and feature objects are lazy DuckDB relations backed by the source Parquet files. Source files are never modified.

Exclude identifiers, addresses, timestamps, capture metadata, and attack annotations from the feature matrix. Preserve them separately for later interpretation. Port numbers, protocol, and ICMP codes/types are categorical: retain them separately for possible encoding.

Remove columns with at most one distinct finite value **across both datasets together**. A feature constant within each dataset but different between datasets remains useful for comparison. Missingness is audited separately.

Requires `duckdb` and `pandas` in the notebook kernel. Restart the kernel before running all cells to release any previously loaded DataFrames.


In [ ]:
from pathlib import Path
import re
import duckdb
import pandas as pd
from IPython.display import display

ROOT = REPO_ROOT
PATHS = {
    '2017': DATA_DIR / 'compatible/CIC2017_shared.parquet',
    '2018': DATA_DIR / 'compatible/CIC2018_shared.parquet',
}
for path in PATHS.values():
    if not path.exists():
        raise FileNotFoundError(path)

def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'

# Limit DuckDB working memory; larger operations may spill to temporary disk.
# This is not a hard cap on total Python process memory.
con = duckdb.connect(config={'memory_limit': '2GB', 'threads': 2})
datasets = {name: con.read_parquet(str(path)) for name, path in PATHS.items()}
assert datasets['2017'].columns == datasets['2018'].columns
row_counts = {name: relation.count('*').fetchall()[0][0]
              for name, relation in datasets.items()}
assert all(row_counts.values()), 'An input dataset is empty'
display(pd.DataFrame({name: {'rows': row_counts[name], 'columns': len(rel.columns)}
                      for name, rel in datasets.items()}).T)


## Review feature roles

The lists below are explicit study choices and can be edited. Labels and `Attempted Category` are annotations, even when stored as integers. Port numbers can describe services, but their numeric distance is unsuitable for ordinary PCA; encoding or grouping them is a later choice. Protocol is kept for the planned one-hot encoding step.

In [ ]:
def normalize(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

metadata_keys = {
    'id', 'flowid', 'index', 'unnamed0',
    'srcip', 'dstip', 'sourceip', 'destinationip',
    'srcipaddress', 'dstipaddress', 'sourceipaddress', 'destinationipaddress',
    'timestamp', 'captureday', 'day', 'date',
    'label', 'attack', 'attacktype', 'attackcategory', 'attemptedcategory',
}
categorical_keys = {
    'protocol', 'srcport', 'dstport', 'sourceport', 'destinationport',
    'icmpcode', 'icmptype',
}
EXTRA_EXCLUDE = []  # Add any further study-specific exclusions here.
columns = list(datasets['2017'].columns)
# Inspect schema types without materializing any rows.
types = {name: dict(zip(rel.columns, rel.types)) for name, rel in datasets.items()}
metadata_columns = [c for c in columns if normalize(c) in metadata_keys or c in EXTRA_EXCLUDE]
categorical_columns = [c for c in columns if normalize(c) in categorical_keys
                       and c not in metadata_columns]
candidates = [c for c in columns if c not in metadata_columns + categorical_columns]
non_numeric = [c for c in candidates
               if not all(types[name][c].id in {'tinyint', 'smallint', 'integer', 'bigint', 'hugeint',
                                          'utinyint', 'usmallint', 'uinteger', 'ubigint',
                                          'uhugeint', 'float', 'double', 'decimal'}
                          for name in datasets)]
numeric_columns = [c for c in candidates if c not in non_numeric]

roles = pd.DataFrame({'column': columns})
roles['role'] = roles['column'].map(lambda c:
    'metadata / excluded' if c in metadata_columns else
    'categorical / encode later' if c in categorical_columns else
    'non-numeric / review' if c in non_numeric else 'numeric candidate')
display(roles)
def project(rel, selected):
    # DuckDB cannot represent a zero-column relation.
    return rel.project(', '.join(map(quote_identifier, selected))) if selected else None

metadata = {name: project(rel, metadata_columns) for name, rel in datasets.items()}
categorical = {name: project(rel, categorical_columns) for name, rel in datasets.items()}
if non_numeric:
    print('Excluded non-numeric columns; review their roles:', non_numeric)

## Remove constants and audit invalid values

Convert positive and negative infinity and NaN to SQL NULL in lazy numeric projections. Constant detection ignores missing values. Columns with no observed finite values are removed. No rows are dropped or imputed.

Compute exact distinct counts one feature at a time, including a joint count over both datasets. This scans the full data and can take time, but avoids keeping all feature values or distinct sets in pandas at once.


In [ ]:
assert numeric_columns, 'No numeric candidate features remain'
numeric = {
    name: rel.project(', '.join(
        f'CASE WHEN isfinite({quote_identifier(c)}) THEN {quote_identifier(c)} '
        f'ELSE NULL END AS {quote_identifier(c)}' for c in numeric_columns))
    for name, rel in datasets.items()
}
for name, rel in numeric.items():
    rel.create_view(f'numeric_{name}', replace=True)

records = []
for col in numeric_columns:
    q = quote_identifier(col)
    counts = {}
    for name in datasets:
        counts[name] = con.execute(
            f'SELECT count(DISTINCT {q}), count(*) - count({q}) '
            f'FROM numeric_{name}'
        ).fetchall()[0]
    joint_unique = con.execute(
        f'SELECT count(DISTINCT value) FROM ('
        f'SELECT {q} AS value FROM numeric_2017 UNION ALL '
        f'SELECT {q} AS value FROM numeric_2018)'
    ).fetchall()[0][0]
    records.append({
        'column': col,
        'unique_2017': counts['2017'][0], 'unique_2018': counts['2018'][0],
        'unique_joint': joint_unique,
        'missing_or_infinite_2017': counts['2017'][1],
        'missing_or_infinite_2018': counts['2018'][1],
        'decision': 'remove: all missing' if joint_unique == 0 else
                    'remove: constant' if joint_unique == 1 else 'keep',
    })
audit = pd.DataFrame(records).set_index('column')
keep = audit.index[audit['decision'].eq('keep')].tolist()
display(audit)
print('Removed columns:', audit.index[~audit['decision'].eq('keep')].tolist())
print('Retained numeric features:', len(keep))
assert keep, 'No variable numeric features remain'
X_numeric = {name: project(rel, keep) for name, rel in numeric.items()}
for name in datasets:
    invalid = ' OR '.join(f'{quote_identifier(c)} IS NULL' for c in keep)
    missing_rows = con.execute(
        f'SELECT count(*) FROM numeric_{name} WHERE {invalid}'
    ).fetchall()[0][0]
    print(f'{name}: ({row_counts[name]:,}, {len(keep)}); '
          f'rows with missing/invalid numeric values: {missing_rows:,}')
assert X_numeric['2017'].columns == X_numeric['2018'].columns


## Ready for the next step

- `X_numeric['2017']` and `X_numeric['2018']`: lazy DuckDB relations with the same numeric features and constants removed.
- `categorical` and `metadata`: lazy projections of the original rows (`None` if no columns have that role).
- `roles` and `audit`: small pandas tables of feature-selection decisions.

Use `X_numeric['2017'].limit(1000).df()` for a bounded preview. Calling `.df()` without a limit materializes the full result and may exhaust RAM again. Keep `con` open while using these relations.

SQL relations have no pandas index or guaranteed row order. For future joins or batch processing, carry a verified stable row identifier with the features and annotations; do not align separately fetched results by position.

These features still need a missing-value policy, shared categorical encoding, and scaling before PCA. Use the same feature ordering, fitted preprocessing, and PCA basis for both datasets. Full-data PCA will need a memory-conscious workflow such as batch processing. No scaling or PCA has been applied here.


In [ ]:
invalid = ' OR '.join(
    f'NOT coalesce(isfinite({quote_identifier(c)}), false)'
    for c in keep
)

bad_rows = datasets['2017'].filter(invalid).df()
display(bad_rows)

# Show which retained numeric features are invalid in each row.
import numpy as np

display(bad_rows[keep].apply(
    lambda row: [
        c for c in keep
        if pd.isna(row[c]) or not np.isfinite(row[c])
    ],
    axis=1,
).rename('invalid_features'))

Get rid of troublesome rows

In [ ]:
# Keep only rows where every retained numeric feature is finite.
valid = ' AND '.join(
    f'coalesce(isfinite({quote_identifier(c)}), false)'
    for c in keep
)

datasets_clean = {
    name: rel.filter(valid)
    for name, rel in datasets.items()
}

# Use these objects for subsequent analysis.
X_clean = {
    name: project(rel, keep)
    for name, rel in datasets_clean.items()
}
metadata = {
    name: project(rel, metadata_columns)
    for name, rel in datasets_clean.items()
}
categorical = {
    name: project(rel, categorical_columns)
    for name, rel in datasets_clean.items()
}

look at zero-traffic flows before PCA

In [ ]:
# Resolve the packet-count column names using the existing normalize() helper.
def find_column(*names):
    matches = [
        c for c in columns
        if normalize(c) in {normalize(name) for name in names}
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one match for {names}; found {matches}")
    return quote_identifier(matches[0])

fwd = find_column('Fwd Packet Length Mean')
bwd = find_column('Bwd Packet Length Mean')

packet_case = f"""
CASE
    WHEN {fwd} IS NULL OR {bwd} IS NULL
         OR NOT isfinite({fwd}) OR NOT isfinite({bwd})
         OR {fwd} < 0 OR {bwd} < 0
        THEN 'Invalid packet counts'
    WHEN {fwd} = 0 AND {bwd} = 0 THEN 'Zero total payload'
    WHEN {fwd} > 0 AND {bwd} = 0 THEN 'Forward only'
    WHEN {fwd} = 0 AND {bwd} > 0 THEN 'Backward only'
    ELSE 'Packets in both directions'
END
"""

summaries = []
for name, rel in datasets_clean.items():
    rel.create_view('packet_check', replace=True)
    rows = con.execute(f"""
        SELECT {packet_case} AS traffic_pattern, count(*) AS rows
        FROM packet_check
        GROUP BY 1
    """).fetchall()  # Fully consume the result.

    summary = pd.DataFrame(rows, columns=['traffic_pattern', 'rows'])
    summary['percent'] = 100 * summary['rows'] / summary['rows'].sum()
    summary['dataset'] = name
    summaries.append(summary)

packet_summary = pd.concat(summaries, ignore_index=True)

display(
    packet_summary.pivot(
        index='traffic_pattern',
        columns='dataset',
        values=['rows', 'percent'],
    ).fillna(0).round(2)
)

In [ ]:
length_checks = [
    'Total Length of Fwd Packet',
    'Total Length of Bwd Packet',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Packet Length Min',
    'Packet Length Max',
]

for year, rel in datasets_clean.items():
    rel.create_view('length_check', replace=True)
    print(f'\n—— {year} ——')

    # 1. Compare the two groups.
    display(con.execute("""
        SELECT
            CASE WHEN "Packet Length Mean" = 0
                 THEN 'Zero mean length'
                 ELSE 'Positive mean length' END AS length_group,
            count(*) AS rows,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2)
                AS percent_of_year,
            median("Total Fwd Packet") AS median_fwd_packets,
            median("Total Bwd packets") AS median_bwd_packets,
            median("Total Length of Fwd Packet") AS median_fwd_length,
            median("Total Length of Bwd Packet") AS median_bwd_length
        FROM length_check
        WHERE "Packet Length Mean" >= 0
        GROUP BY 1
    """).df())

    # 2. Within zero-mean flows, do related length fields also equal zero?
    checks = ', '.join(
        f'100.0 * avg(CASE WHEN {quote_identifier(c)} = 0 '
        f'THEN 1.0 ELSE 0.0 END) AS {quote_identifier(c)}'
        for c in length_checks
    )
    agreement = con.execute(f"""
        SELECT {checks}
        FROM length_check
        WHERE "Packet Length Mean" = 0
    """).df().T
    agreement.columns = ['percent_zero_among_zero_mean_flows']
    display(agreement.round(2))

    # 3. Protocol/label breakdown, including the zero rate within each group.
    display(con.execute("""
        SELECT
            "Protocol", "Label",
            count(*) AS total_rows,
            count(*) FILTER (WHERE "Packet Length Mean" = 0)
                AS zero_mean_rows,
            round(
                100.0 * count(*) FILTER (WHERE "Packet Length Mean" = 0)
                / count(*), 2
            ) AS percent_zero_within_protocol_label
        FROM length_check
        GROUP BY 1, 2
        ORDER BY zero_mean_rows DESC
    """).df())

    # 4. Inspect a few complete examples without loading the whole group.
    display(con.execute("""
        SELECT
            "Protocol", "Label",
            "Total Fwd Packet", "Total Bwd packets",
            "Packet Length Mean",
            "Total Length of Fwd Packet", "Total Length of Bwd Packet",
            "Fwd Header Length", "Bwd Header Length",
            "SYN Flag Count", "ACK Flag Count", "RST Flag Count",
            "Flow Duration"
        FROM length_check
        WHERE "Packet Length Mean" = 0
        LIMIT 10
    """).df())

look at log and lin distribs for rescaling before PCA analysis

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

SAMPLE_SIZE = 30_000
BINS = 60
FEATURES_PER_PAGE = 4
OUTPUT = ARTIFACT_DIR / "continuous_features_linear_vs_log.pdf"

# Edit this list if some retained numeric columns are not continuous
# measurements you want to inspect.
plot_features = list(keep)

# Sample inside DuckDB before bringing data into Python.
samples = {}
selection = ', '.join(map(quote_identifier, plot_features))

for year, rel in datasets_clean.items():
    rel.create_view('plot_source', replace=True)
    samples[year] = con.execute(f"""
        SELECT {selection}
        FROM plot_source
        USING SAMPLE reservoir ({SAMPLE_SIZE} ROWS) REPEATABLE (42)
    """).df()

colors = {'2017': 'tab:blue', '2018': 'tab:orange'}


def draw_histograms(ax, values_by_year):
    nonempty = [v for v in values_by_year.values() if len(v)]
    if not nonempty:
        ax.text(
            0.5, 0.5, 'No finite nonzero values in sample',
            ha='center', va='center', transform=ax.transAxes
        )
        return

    # Shared bin boundaries make the two years comparable.
    edges = np.histogram_bin_edges(np.concatenate(nonempty), bins=BINS)

    for year, values in values_by_year.items():
        if not len(values):
            continue
        counts, _ = np.histogram(values, bins=edges)
        ax.stairs(
            100 * counts / len(values),
            edges,
            label=year,
            color=colors.get(year),
            linewidth=1.3,
        )

    ax.set_ylabel('% of nonzero sample / bin')
    ax.legend()
    ax.grid(alpha=0.2)


with PdfPages(OUTPUT) as pdf:
    for start in range(0, len(plot_features), FEATURES_PER_PAGE):
        page_features = plot_features[start:start + FEATURES_PER_PAGE]
        fig, axes = plt.subplots(
            len(page_features), 2,
            figsize=(15, 3.5 * len(page_features)),
            squeeze=False,
            constrained_layout=True,
        )

        for row, feature in enumerate(page_features):
            values_by_year = {}
            zero_notes = []

            for year, sample in samples.items():
                values = sample[feature].to_numpy(dtype=float)
                values = values[np.isfinite(values)]
                zero_pct = 100 * np.mean(values == 0) if len(values) else np.nan
                zero_notes.append(f'{year}: {zero_pct:.1f}% zeros')
                values_by_year[year] = values[values != 0]

            has_negatives = any(
                np.any(values < 0)
                for values in values_by_year.values()
            )

            if has_negatives:
                transformed = {
                    year: np.sign(values) * np.log1p(np.abs(values))
                    for year, values in values_by_year.items()
                }
                transform_label = 'sign(x) × ln(1 + |x|)'
            else:
                transformed = {
                    year: np.log10(values)
                    for year, values in values_by_year.items()
                }
                transform_label = 'log10(x)'

            draw_histograms(axes[row, 0], values_by_year)
            draw_histograms(axes[row, 1], transformed)

            axes[row, 0].set_title(
                f'{feature} — linear\n' + ' | '.join(zero_notes),
                fontsize=10,
            )
            axes[row, 1].set_title(
                f'{feature} — {transform_label}',
                fontsize=10,
            )
            axes[row, 0].set_xlabel('Original value')
            axes[row, 1].set_xlabel(transform_label)

        fig.suptitle(
            'Nonzero feature distributions — sampled rows, shared bins',
            fontsize=14,
        )
        pdf.savefig(fig)

        # Show the first page in the notebook as a preview.
        if start == 0:
            plt.show()
        plt.close(fig)

print(f"Saved all {len(plot_features)} features to: {OUTPUT.resolve()}")

## Log1p → StandardScaler → shared PCA (continuous measurements)

Run the preparation cells through `datasets_clean` first. Use those same cleaned rows, including zero-length flows. Packet/byte counts, durations, rates, lengths, and window sizes are treated as quantitative measurements; flag indicators/counts are excluded from this continuous-feature analysis, as are protocol, ports, ICMP codes, identifiers, and annotations. Edit `PCA_EXTRA_EXCLUDE` if required.

**Category means the original `Label`**, not a harmonized attack family. Labels are never inputs to PCA. Different names across years remain different categories.

Fit a single `StandardScaler` to log-transformed rows pooled across the two years, giving every row equal weight. Thus the larger dataset has more influence. Both datasets use the same scaler and PC axes. Separate fits would rotate the axes and prevent direct coordinate comparisons.

Non-finite and negative values are reported per feature and year, then treated as missing in this analysis: negative sentinel values such as −1 cannot be passed to ordinary log1p. Apply `ln(1+x)` to finite nonnegative values. Fit `StandardScaler.partial_fit` column by column on observed entries, skipping all-missing batches; filling missing standardized entries with zero is equivalent to pooled-mean imputation **after log1p**. No signed-log transform or clipping is used. Inspect the audit before interpreting the results.

All fitting and variance summaries use full cleaned datasets in bounded batches. Only scatter plots use capped random samples per label. No source files are changed. Requires NumPy, scikit-learn, matplotlib, and PyArrow as well as the preparation dependencies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

BATCH_SIZE = 32_768
PLOT_ROWS_PER_CATEGORY = 1_000
RANDOM_SEED = 42
PCA_EXTRA_EXCLUDE = []
CATEGORY_COLUMN = next(c for c in columns if normalize(c) == 'label')
flag_columns = [c for c in keep if 'flag' in normalize(c)]
pca_features = [c for c in keep if c not in flag_columns + PCA_EXTRA_EXCLUDE]
assert len(pca_features) >= 2
print('Excluded flag fields:', flag_columns)
print('Continuous/quantitative features:', len(pca_features))

# Read annotations and features together; never align independent SQL results by position.
for year, rel in datasets_clean.items():
    rel.create_view(f'pca_source_{year}', replace=True)

def pca_batches(year):
    selected = ', '.join(map(quote_identifier, pca_features + [CATEGORY_COLUMN]))
    relation = con.sql(f'SELECT {selected} FROM pca_source_{year}')
    if hasattr(relation, 'to_arrow_reader'):
        reader = relation.to_arrow_reader(batch_size=BATCH_SIZE)
    else:
        reader = relation.fetch_record_batch(BATCH_SIZE)
    for batch in reader:
        frame = batch.to_pandas()
        values = frame[pca_features].to_numpy(dtype=np.float64, copy=True)
        labels = frame[CATEGORY_COLUMN].astype('string').fillna('<missing label>').to_numpy(dtype=str)
        yield values, labels

def log_measurements(values):
    valid = np.isfinite(values) & (values >= 0)
    logged = np.full_like(values, np.nan)
    np.log1p(values, out=logged, where=valid)
    return logged


In [ ]:
# Fit each column on observed log values, skipping all-missing batches.
# This avoids NaN variance updates when a whole batch contains sentinels.
feature_scalers = [StandardScaler() for _ in pca_features]
log_audit_rows = []
clean_counts = {}
for year in datasets_clean:
    negative = np.zeros(len(pca_features), dtype=np.int64)
    invalid = negative.copy()
    n = 0
    for values, labels in pca_batches(year):
        negative += ((values < 0) & np.isfinite(values)).sum(axis=0)
        invalid += (~np.isfinite(values)).sum(axis=0)
        logged = log_measurements(values)
        for j, fitted in enumerate(feature_scalers):
            observed = logged[np.isfinite(logged[:, j]), j]
            if len(observed):
                fitted.partial_fit(observed[:, None])
        n += len(values)
    clean_counts[year] = n
    for j, feature in enumerate(pca_features):
        log_audit_rows.append({'dataset': year, 'feature': feature,
                              'rows': n, 'negative_to_missing': negative[j],
                              'nonfinite_to_missing': invalid[j]})
    print(f'{year}: {n:,} cleaned rows used; {row_counts[year] - n:,} rows removed by earlier cleaning')

# Assemble the independently fitted column statistics into one shared scaler.
scaler = StandardScaler()
scaler.n_features_in_ = len(pca_features)
scaler.mean_ = np.array([getattr(s, 'mean_', [np.nan])[0] for s in feature_scalers])
scaler.var_ = np.array([getattr(s, 'var_', [np.nan])[0] for s in feature_scalers])
scaler.scale_ = np.array([getattr(s, 'scale_', [np.nan])[0] for s in feature_scalers])
scaler.n_samples_seen_ = np.array([getattr(s, 'n_samples_seen_', 0) for s in feature_scalers])
log_audit = pd.DataFrame(log_audit_rows)
display(log_audit.query('negative_to_missing > 0 or nonfinite_to_missing > 0'))
# Recheck constants after the log/missing-value policy.
active_features = np.isfinite(scaler.var_) & (scaler.var_ > 0)
assert active_features.sum() >= 2, 'Fewer than two variable features remain after log1p'
feature_names = np.asarray(pca_features)[active_features]
print('Removed after log/missing policy:', np.asarray(pca_features)[~active_features].tolist())
scaling_summary = pd.DataFrame({
    'feature': pca_features, 'log_mean': scaler.mean_, 'log_std': np.sqrt(scaler.var_),
    'observed_rows': np.broadcast_to(scaler.n_samples_seen_, (len(pca_features),)),
    'used_in_pca': active_features,
})
display(scaling_summary)


def standardize(values):
    # Fill in log space, then call the actual StandardScaler transform.
    logged = log_measurements(values)
    logged = np.where(np.isnan(logged), scaler.mean_[None, :], logged)
    z = scaler.transform(logged)[:, active_features]
    assert np.isfinite(z).all()
    return z


### Exact PCA from batch covariance

Accumulate centered cross-products in float64 with stable batch moment merging. Diagonalizing the pooled covariance gives ordinary full PCA, without loading millions of rows into a dense matrix or using an approximate incremental PCA. The only large objects are individual batches; covariance matrices are feature × feature.

For each dataset and label we retain its count, centroid, and within-group covariance. Projecting those covariances onto the **shared** PCs measures the fraction of that group's own centered variation on each PC. This differs from a group's offset from the pooled center; both are reported below.

In [ ]:
class Moments:
    def __init__(self, p):
        self.n = 0
        self.mean = np.zeros(p)
        self.m2 = np.zeros((p, p))

    def merge(self, other):
        if other.n == 0:
            return
        total = self.n + other.n
        delta = other.mean - self.mean
        self.m2 += other.m2 + np.outer(delta, delta) * (self.n * other.n / total)
        self.mean += delta * (other.n / total)
        self.n = total

    def update(self, z):
        if len(z) == 0:
            return
        batch = Moments(z.shape[1])
        batch.n = len(z)
        batch.mean = z.mean(axis=0)
        centered = z - batch.mean
        batch.m2 = centered.T @ centered
        self.merge(batch)

p = len(feature_names)
year_moments = {year: Moments(p) for year in datasets_clean}
category_moments = {}
plot_reservoir = {}
rng = np.random.default_rng(RANDOM_SEED)
for year in datasets_clean:
    for values, labels in pca_batches(year):
        z = standardize(values)
        year_moments[year].update(z)
        for label in np.unique(labels):
            group = z[labels == label]
            key = (year, label)
            category_moments.setdefault(key, Moments(p)).update(group)
            # Random-priority reservoir: uniform sample within each year/label,
            # with bounded storage independent of the full dataset size.
            priorities = rng.random(len(group))
            old_keys, old_z = plot_reservoir.get(key, (np.empty(0), np.empty((0, p))))
            priorities = np.concatenate([old_keys, priorities])
            points = np.concatenate([old_z, group])
            if len(points) > PLOT_ROWS_PER_CATEGORY:
                chosen = np.argpartition(priorities, PLOT_ROWS_PER_CATEGORY - 1)[:PLOT_ROWS_PER_CATEGORY]
                priorities, points = priorities[chosen], points[chosen]
            plot_reservoir[key] = (priorities, points)
    print(f'Accumulated full-data covariance and label summaries: {year}')

pooled = Moments(p)
for year, moments in year_moments.items():
    assert moments.n == clean_counts[year]
    pooled.merge(moments)
assert pooled.n > 1
covariance = pooled.m2 / (pooled.n - 1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = np.maximum(eigenvalues[order], 0)
components = eigenvectors[:, order].T
# Deterministic signs; signs have no effect on explained variance.
for component in components:
    if component[np.argmax(np.abs(component))] < 0:
        component *= -1
pc_names = [f'PC{i + 1}' for i in range(p)]
explained_ratio = eigenvalues / eigenvalues.sum()
pooled_scree = pd.DataFrame({
    'PC': pc_names, 'eigenvalue': eigenvalues,
    'variance_percent': 100 * explained_ratio,
    'cumulative_percent': 100 * np.cumsum(explained_ratio),
})
assert np.allclose(components @ components.T, np.eye(p))
assert np.isclose(eigenvalues.sum(), np.trace(covariance))
display(pooled_scree.head(20))
print('PCs to reach 90% / 95%:',
      np.searchsorted(np.cumsum(explained_ratio), [0.90, 0.95]) + 1)


In [ ]:
def group_variance(moments):
    # Common denominator cancels in ratios: M2 is sufficient and robust.
    axis_ss = np.maximum(np.einsum('ij,jk,ik->i', components, moments.m2, components), 0)
    total_ss = np.trace(moments.m2)
    ratios = axis_ss / total_ss if total_ss > 0 else np.full(p, np.nan)
    centroid = (moments.mean - pooled.mean) @ components.T
    return axis_ss, ratios, centroid

variance_rows, summary_rows = [], []
all_groups = [('dataset', year, '<all>', moments)
              for year, moments in year_moments.items()]
all_groups += [('category', year, label, moments)
               for (year, label), moments in sorted(category_moments.items())]
for level, year, label, moments in all_groups:
    ss, ratios, centroid = group_variance(moments)
    within = np.trace(moments.m2)
    offset = moments.n * np.sum(centroid ** 2)
    summary_rows.append({
        'level': level, 'dataset': year, 'category': label, 'rows': moments.n,
        'PC1_within_percent': 100 * ratios[0],
        'PC2_within_percent': 100 * ratios[1],
        'PC1_PC2_within_percent': 100 * ratios[:2].sum(),
        'PC1_centroid': centroid[0], 'PC2_centroid': centroid[1],
        'within_SS': within, 'offset_SS': offset,
        'pooled_SS_percent': 100 * (within + offset) / np.trace(pooled.m2),
    })
    for j, pc in enumerate(pc_names):
        variance_rows.append({
            'level': level, 'dataset': year, 'category': label, 'PC': pc,
            'rows': moments.n, 'within_variance_percent': 100 * ratios[j],
            'within_cumulative_percent': 100 * np.cumsum(ratios)[j],
            'centroid': centroid[j],
        })
variance_by_group = pd.DataFrame(variance_rows)
group_summary = pd.DataFrame(summary_rows)
dataset_summary = group_summary.query("level == 'dataset'").set_index('dataset')
category_summary = group_summary.query("level == 'category'").set_index(['dataset', 'category'])
display(dataset_summary.drop(columns=['level', 'category']))
display(category_summary.drop(columns='level'))

# Check within-group spread + displacement accounts for all pooled variation.
for level in ['dataset', 'category']:
    totals = group_summary.query('level == @level')
    assert np.isclose((totals.within_SS + totals.offset_SS).sum(), np.trace(pooled.m2))

# Full per-PC breakdowns; these tables include every component, not just PC1/PC2.
dataset_variance = variance_by_group.query("level == 'dataset'").pivot(
    index='PC', columns='dataset', values='within_variance_percent').reindex(pc_names)
category_variance = variance_by_group.query("level == 'category'").pivot(
    index=['dataset', 'category'], columns='PC', values='within_variance_percent').reindex(columns=pc_names)
display(dataset_variance.head(15))
display(category_variance.iloc[:, :10])


**Reading these tables:** `within_variance_percent` is the share of that dataset/category's own variation lying along a shared PC. The shares sum to 100% over all PCs for groups with nonzero variance; one-row or zero-spread groups have undefined shares (NaN). A category can be far from the pooled center yet have little internal spread. `PC1_centroid` / `PC2_centroid` show that displacement. `pooled_SS_percent` includes both internal spread and displacement, weighted by row count; it sums to 100% over datasets, or separately over all dataset/category groups. These are descriptive results for the cleaned data and chosen preprocessing, not evidence by themselves of attack separability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
x = np.arange(1, p + 1)
axes[0].plot(x, 100 * explained_ratio, 'o-', ms=3, label='Pooled')
axes[1].plot(x, 100 * np.cumsum(explained_ratio), label='Pooled')
for year in datasets_clean:
    ratios = group_variance(year_moments[year])[1]
    axes[0].plot(x, 100 * ratios, label=f'{year}, shared PCs')
    axes[1].plot(x, 100 * np.cumsum(ratios), label=f'{year}, shared PCs')
for ax in axes:
    ax.set_xlabel('Shared principal component')
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel('Within-dataset variance (%)')
axes[1].set_ylabel('Cumulative within-dataset variance (%)')
axes[0].set_title('Variance along each common axis')
axes[1].set_title('Cumulative variance captured by common axes')
plt.show()

heat = category_variance.iloc[:, :min(10, p)]
fig, ax = plt.subplots(figsize=(12, max(5, len(heat) * 0.32)), constrained_layout=True)
image = ax.imshow(np.ma.masked_invalid(heat.to_numpy()), aspect='auto', cmap='viridis', vmin=0)
ax.set_yticks(np.arange(len(heat)), [f'{year} | {label}' for year, label in heat.index])
ax.set_xticks(np.arange(heat.shape[1]), heat.columns)
ax.set_title('Within-category variance captured by shared PCs (%) — full data')
fig.colorbar(image, ax=ax, label='Within-category variance (%)')
plt.show()

# Loadings: feature weights defining the common PC directions.
loadings = pd.DataFrame(components.T, index=feature_names, columns=pc_names)
for pc in pc_names[:2]:
    top = loadings[pc].abs().nlargest(12).index
    print(f'{pc}: largest absolute feature weights')
    display(loadings.loc[top, [pc]])


In [ ]:
# Same colors and axis limits for both datasets. Sampling is per category:
# plotted category frequencies do NOT represent their population prevalence.
labels_all = sorted({label for year, label in plot_reservoir})
cmap = plt.get_cmap('tab20' if len(labels_all) <= 20 else 'turbo')
category_colors = {label: cmap(i / max(1, len(labels_all) - 1))
                   for i, label in enumerate(labels_all)}
score_samples = {}
for (year, label), (_, z) in plot_reservoir.items():
    scores = (z - pooled.mean) @ components[:2].T
    score_samples[(year, label)] = scores

fig, axes = plt.subplots(1, len(datasets_clean), figsize=(16, 7),
                         sharex=True, sharey=True, squeeze=False)
for ax, year in zip(axes[0], datasets_clean):
    for label in labels_all:
        points = score_samples.get((year, label))
        if points is None:
            continue
        ax.scatter(points[:, 0], points[:, 1], s=5, alpha=0.35,
                   color=category_colors[label], rasterized=True)
    # Crosses are full-data category centroids, not sample centroids.
    for (group_year, label), moments in category_moments.items():
        if group_year == year:
            centroid = group_variance(moments)[2]
            ax.scatter(*centroid[:2], marker='x', s=55, linewidths=1.5,
                       color=category_colors[label])
    within = group_variance(year_moments[year])[1][:2].sum()
    ax.set_title(f'{year}: {clean_counts[year]:,} rows\nPC1–PC2 capture {100 * within:.1f}% of within-year variance')
    ax.set_xlabel(f'PC1 (pooled: {100 * explained_ratio[0]:.1f}%)')
    ax.set_ylabel(f'PC2 (pooled: {100 * explained_ratio[1]:.1f}%)')
    ax.grid(alpha=0.2)
from matplotlib.lines import Line2D
handles = [Line2D([], [], marker='o', linestyle='', color=category_colors[label],
                  label=label, markersize=5) for label in labels_all]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0),
           ncol=min(4, len(handles)), fontsize=8, title='Original Label (× = full-category centroid)')
legend_height = 0.08 + 0.035 * int(np.ceil(len(handles) / min(4, len(handles))))
fig.subplots_adjust(bottom=min(0.5, legend_height), top=0.85)
fig.suptitle('Shared log1p + StandardScaler PCA\nUniform sample per year/category; shared axes; no percentile trimming')
plt.show()
display(pd.DataFrame([
    {'dataset': year, 'category': label, 'full_rows': category_moments[(year, label)].n,
     'plotted_rows': len(points)}
    for (year, label), points in sorted(score_samples.items())
]))
